# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tabassumrafiq/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

## 1. Method choice and why

My lane is content review prioritization, so the modeling problem is a
binary classification problem that can also be used as a ranking problem.

I will start with Logistic Regression because it is simple, reproducible,
and provides probabilities that can be used to rank content items.

I will also test Random Forest as a stronger non-linear comparison. I will
not prefer the more complex model unless it improves the measured ranking
performance on the held-out data.

The target is `is_declining_label`.

I will not use `trend_pct` or `trend_direction` because they are used to
derive the label. I will also exclude future-window fields such as
`impressions_last30`, `clicks_last30`, and `avg_position_last30` because
they would not be available at prediction time.

The goal is decision-support, not proof that a content refresh will cause
an improvement.

In [18]:
import pandas as pd
import numpy as np

from pathlib import Path

from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

RANDOM_STATE = 42

print("Libraries loaded.")
print("Random seed:", RANDOM_STATE)

Libraries loaded.
Random seed: 42


In [19]:
from pathlib import Path

DATA_PATH = Path("/content/content_refresh_anonymized.csv")

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found at {DATA_PATH}. "
        "Run the notebook from the repository root."
    )

data = pd.read_csv(DATA_PATH)

print("Dataset shape:", data.shape)
print("\nColumns:")
print(data.columns.tolist())

display(data.head())

Dataset shape: (30000, 44)

Columns:
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [21]:
data["is_declining_label"] = data["trend_direction"] == "down"

print("Target distribution:")
display(data["is_declining_label"].value_counts(dropna=False))

print("\nTarget rate:")
print(data["is_declining_label"].mean())

print("\nMissing target values:")
print(data["is_declining_label"].isna().sum())

Target distribution:


,count
is_declining_label,
True,16262
False,13738



Target rate:
0.5420666666666667

Missing target values:
0


In [22]:
feature_candidates = [
    "gsc_impressions",
    "avg_position",
    "clicks",
    "ctr",
    "engagement_rate",
    "scroll_rate",
    "word_count"
]

available_features = [
    col for col in feature_candidates
    if col in data.columns
]

print("Available candidate features:")
print(available_features)

Available candidate features:
['avg_position', 'ctr', 'engagement_rate', 'scroll_rate', 'word_count']


In [23]:
future_or_label_terms = [
    "trend",
    "last30",
    "future",
    "outcome",
    "label",
    "needs_refresh"
]

possible_leaks = [
    col for col in available_features
    if any(term in col.lower() for term in future_or_label_terms)
]

print("Possible leakage columns:")
print(possible_leaks)

assert len(possible_leaks) == 0, "Potential leakage detected!"

print("\nLeakage check passed.")

Possible leakage columns:
[]

Leakage check passed.


In [24]:
model_df = data[
    ["client_id", "content_id", "is_declining_label"] + available_features
].copy()

model_df = model_df.dropna(subset=["is_declining_label"])

X = model_df[available_features].copy()
y = model_df["is_declining_label"].astype(int)
groups = model_df["client_id"]

print("Rows used:", len(model_df))
print("Features:", available_features)
print("Target classes:", sorted(y.unique()))
print("Base rate:", round(y.mean(), 4))

Rows used: 30000
Features: ['avg_position', 'ctr', 'engagement_rate', 'scroll_rate', 'word_count']
Target classes: [np.int64(0), np.int64(1)]
Base rate: 0.5421


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

## 2. Split design

I will use a grouped train/test split by `client_id`.

Client identifiers are not used as predictive features. They are used only
to prevent content from the same client appearing in both training and test
sets.

This is more conservative than a random row split because content from the
same client can share characteristics.

The split is fixed with random seed 42 so the experiment is reproducible.

In [25]:
gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=RANDOM_STATE
)

train_idx, test_idx = next(
    gss.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx].copy()
X_test = X.iloc[test_idx].copy()

y_train = y.iloc[train_idx].copy()
y_test = y.iloc[test_idx].copy()

groups_train = groups.iloc[train_idx]
groups_test = groups.iloc[test_idx]

print("Train rows:", len(X_train))
print("Test rows:", len(X_test))

print("\nTrain clients:", groups_train.nunique())
print("Test clients:", groups_test.nunique())

print("\nClient overlap:")
print(len(set(groups_train) & set(groups_test)))

assert len(set(groups_train) & set(groups_test)) == 0

print("Grouped split check passed.")

Train rows: 23837
Test rows: 6163

Train clients: 25
Test clients: 7

Client overlap:
0
Grouped split check passed.


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

## 3. Train + compare vs my baseline

Logistic Regression is the first learned model because it is simple and
interpretable. Its predicted probability will be used as the ranking score.

Missing numeric values are median-imputed inside the pipeline, and features
are standardized using only the training data.

The model is evaluated on the held-out test set.

In [26]:
logistic_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        max_iter=1000,
        random_state=RANDOM_STATE
    ))
])

logistic_model.fit(X_train, y_train)

logistic_prob = logistic_model.predict_proba(X_test)[:, 1]
logistic_pred = (logistic_prob >= 0.5).astype(int)

print("Logistic Regression trained.")

Logistic Regression trained.


In [27]:
rf_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", RandomForestClassifier(
        n_estimators=200,
        max_depth=6,
        random_state=RANDOM_STATE,
        n_jobs=-1
    ))
])

rf_model.fit(X_train, y_train)

rf_prob = rf_model.predict_proba(X_test)[:, 1]
rf_pred = (rf_prob >= 0.5).astype(int)

print("Random Forest trained.")

Random Forest trained.


In [28]:
def precision_at_k(scores, labels, k):
    scores = np.asarray(scores)
    labels = np.asarray(labels)

    k = min(k, len(scores))

    order = np.argsort(-scores)[:k]

    return labels[order].mean()

In [30]:
baseline_score = (
    X_test["ctr"].fillna(
        X_train["ctr"].median()
    )
    / (
        X_test["avg_position"].replace(0, np.nan).fillna(
            X_train["avg_position"].median()
        )
    )
)

baseline_score = baseline_score.to_numpy()

In [31]:
K_values = [10, 20, 50]

comparison_rows = []

for k in K_values:
    comparison_rows.append({
        "method": "Week-4 baseline",
        "K": k,
        "precision_at_k": precision_at_k(
            baseline_score,
            y_test.to_numpy(),
            k
        )
    })

    comparison_rows.append({
        "method": "Logistic Regression",
        "K": k,
        "precision_at_k": precision_at_k(
            logistic_prob,
            y_test.to_numpy(),
            k
        )
    })

    comparison_rows.append({
        "method": "Random Forest",
        "K": k,
        "precision_at_k": precision_at_k(
            rf_prob,
            y_test.to_numpy(),
            k
        )
    })

comparison_df = pd.DataFrame(comparison_rows)

print("Base rate:", round(y_test.mean(), 4))

display(comparison_df)

Base rate: 0.511


,method,K,precision_at_k
0,Week-4 baseline,10,0.30
1,Logistic Regression,10,0.30
2,Random Forest,10,0.50
3,Week-4 baseline,20,0.50
4,Logistic Regression,20,0.30
5,Random Forest,20,0.55
6,Week-4 baseline,50,0.48
7,Logistic Regression,50,0.40
8,Random Forest,50,0.56


In [32]:
classification_results = pd.DataFrame([
    {
        "model": "Logistic Regression",
        "accuracy": accuracy_score(y_test, logistic_pred),
        "precision": precision_score(y_test, logistic_pred, zero_division=0),
        "recall": recall_score(y_test, logistic_pred, zero_division=0),
        "f1": f1_score(y_test, logistic_pred, zero_division=0)
    },
    {
        "model": "Random Forest",
        "accuracy": accuracy_score(y_test, rf_pred),
        "precision": precision_score(y_test, rf_pred, zero_division=0),
        "recall": recall_score(y_test, rf_pred, zero_division=0),
        "f1": f1_score(y_test, rf_pred, zero_division=0)
    }
])

display(classification_results)

,model,accuracy,precision,recall,f1
0,Logistic Regression,0.527178,0.520262,0.958082,0.674341
1,Random Forest,0.539834,0.528326,0.926961,0.673046


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

## 4. Errors and interpretation

The ranking metric is more important for this lane than accuracy alone
because the operational question is which items should be reviewed first.

I will inspect concrete false positives and false negatives before drawing
conclusions.

A false positive is an item the model ranked as declining but whose observed
label is 0.

A false negative is an item whose observed label is 1 but which received a
lower model probability.

The feature interpretation is directional: feature importance does not prove
causality.

In [33]:
error_review = model_df.iloc[test_idx][
    ["client_id", "content_id", "is_declining_label"] + available_features
].copy()

error_review["model_probability"] = logistic_prob
error_review["model_prediction"] = logistic_pred

false_positives = error_review[
    (error_review["is_declining_label"] == 0) &
    (error_review["model_prediction"] == 1)
].copy()

false_negatives = error_review[
    (error_review["is_declining_label"] == 1) &
    (error_review["model_prediction"] == 0)
].copy()

print("False positives:", len(false_positives))
print("False negatives:", len(false_negatives))

print("\nExample false positives:")
display(false_positives.head(3))

print("\nExample false negatives:")
display(false_negatives.head(3))

False positives: 2782
False negatives: 132

Example false positives:


,client_id,content_id,is_declining_label,avg_position,ctr,engagement_rate,scroll_rate,word_count,model_probability,model_prediction
26,client_4e07408562,content_72c5c2d73e5a,False,30.0,0.12,0.0,11.11,2686.0,0.526846,1
36,client_f369cb89fc,content_bce275871a25,False,5.4,1.35,0.0,0.00,2510.0,0.529404,1
56,client_f369cb89fc,content_dcebfd222b10,False,4.6,0.00,0.0,50.00,3158.0,0.575284,1



Example false negatives:


,client_id,content_id,is_declining_label,avg_position,ctr,engagement_rate,scroll_rate,word_count,model_probability,model_prediction
60,client_f369cb89fc,content_b9104a222d01,True,6.2,4.00,0.0,0.00,2492.0,0.493102,0
128,client_8527a891e2,content_d660fc1fba2c,True,25.5,1.45,0.0,0.00,2020.0,0.491584,0
614,client_f369cb89fc,content_0c0d41e86269,True,54.5,0.00,0.0,33.33,2432.0,0.497997,0


In [34]:
logistic_coefficients = pd.DataFrame({
    "feature": available_features,
    "coefficient": logistic_model.named_steps["model"].coef_[0]
})

logistic_coefficients["absolute_coefficient"] = (
    logistic_coefficients["coefficient"].abs()
)

logistic_coefficients = logistic_coefficients.sort_values(
    "absolute_coefficient",
    ascending=False
)

print("Top features by absolute Logistic Regression coefficient:")
display(logistic_coefficients.head(3))

Top features by absolute Logistic Regression coefficient:


,feature,coefficient,absolute_coefficient
1,ctr,-0.191645,0.191645
4,word_count,0.170828,0.170828
0,avg_position,-0.064546,0.064546


In [35]:
rf_importance = pd.DataFrame({
    "feature": available_features,
    "importance": rf_model.named_steps["model"].feature_importances_
}).sort_values(
    "importance",
    ascending=False
)

print("Random Forest feature importance:")
display(rf_importance)

Random Forest feature importance:


,feature,importance
0,avg_position,0.551091
4,word_count,0.224009
1,ctr,0.097863
3,scroll_rate,0.094622
2,engagement_rate,0.032415


In [36]:
all_model_columns = available_features

leakage_terms = [
    "trend",
    "last30",
    "future",
    "outcome",
    "label",
    "needs_refresh",
    "product_flag"
]

model_leaks = [
    col for col in all_model_columns
    if any(term in col.lower() for term in leakage_terms)
]

print("Possible leakage columns:")
print(model_leaks)

assert len(model_leaks) == 0

print("\nLeakage check passed.")
print("No obvious label-derived, future-window, or product-flag fields are used.")

Possible leakage columns:
[]

Leakage check passed.
No obvious label-derived, future-window, or product-flag fields are used.


In [37]:
best_by_k = (
    comparison_df
    .sort_values(["K", "precision_at_k"], ascending=[True, False])
    .groupby("K")
    .head(1)
)

print("Best method at each K:")
display(best_by_k)

print("\nBase rate:", round(y_test.mean(), 4))

Best method at each K:


,method,K,precision_at_k
2,Random Forest,10,0.50
5,Random Forest,20,0.55
8,Random Forest,50,0.56



Base rate: 0.511


## Interpretation

The baseline and learned models were evaluated on the same held-out test
split using precision@K.

The base rate is reported because precision@K should be interpreted relative
to the prevalence of the positive outcome.

Logistic Regression provides an interpretable probability ranking. Random
Forest provides a non-linear comparison.

The comparison table should be read as measured decision-support performance,
not as evidence that the model causes content improvement.

If a learned model does not consistently improve precision@K over the
transparent Week-4 baseline, the baseline remains useful and additional model
complexity is not justified.

The error review shows that some cases remain difficult. False positives can
occur when historical signals resemble declining content without the observed
label being positive, while false negatives can occur when the available
historical features do not provide enough signal.

Feature coefficients and Random Forest importance are directional indicators
of what the fitted models used. They should not be interpreted as causal
effects.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

## Self-check

- [x] Method choice is explained.
- [x] Client-grouped train/test split is used.
- [x] Random seed is fixed at 42.
- [x] Future-window and label-derived fields are excluded.
- [x] Client and content IDs are not used as predictive features.
- [x] Logistic Regression is trained first.
- [x] Random Forest is used as a stronger comparison.
- [x] Week-4 baseline is evaluated on the same held-out test data.
- [x] Precision@K is reported for the baseline and learned models.
- [x] Base rate is reported.
- [x] False positives and false negatives are inspected.
- [x] Feature interpretation is reported as directional rather than causal.
- [x] Leakage check is included.
- [ ] Runtime → Run all completed without errors.
- [ ] The final comparison numbers were inspected.
- [ ] `work/notebooks/w05_model.ipynb` was committed to the repository.